## 1. Read data

### 1.1. The dataset of sentences

In [1]:
import pandas as pd
sentence_df = pd.read_excel("/workspace/mijnidbcoachnlp/data/analysis_datasets/translated_sentence_data.xlsx")

In [2]:
sentences = sentence_df["Clean_Sentence"].to_list()

### 1.2. Import list of stopwords

In [3]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/mijnidbcoachnlp/data/analysis_datasets/stopwords.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

extra_list = [
    'maandag', 'dinsdag', 'woensdag', 'donderdag', 'vrijdag', 'zaterdag', 'zondag',
    'januari', 'februari', 'maart', 'april', 'mei', 'juni', 'juli', 'augustus', 'september', 'oktober', 'november', 'december',
    'jan', 'feb', 'mrt', 'apr', 'mei', 'jun', 'jul', 'aug', 'sep', 'okt', 'nov', 'dec'
]

dutch_stopwords.extend(extra_list)

## 2. Download the embedding model BERTje

### 2.1 Download BERTje

In [4]:
from transformers.pipelines import pipeline
from transformers import AutoTokenizer, AutoModel, TFAutoModel
embedding_model = pipeline("feature-extraction", model="GroNLP/bert-base-dutch-cased")

Some weights of BertModel were not initialized from the model checkpoint at GroNLP/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 2.2 Pre-calculate and save sentence embeddings (skip if there are saved embeddings and go to 2.3)

In [ ]:
# pre-calculate the embeddings of the dutch sentences
import numpy as np

embeddings_nl = embedding_model(sentences, truncation=True, padding=True)
sentence_embeddings_nl = np.array([np.mean(embedding, axis=1).flatten() for embedding in embeddings_nl])

In [ ]:
# save the sentence embeddings
import numpy as np
np.save('/workspace/mijnidbcoachnlp/saved_models/sentence_embeddings_bertje_nl.npy', sentence_embeddings_nl)

### 2.3 Load saved embeddings

In [5]:
# load the save model
import numpy as np

loaded_embeddings_nl = np.load('/workspace/mijnidbcoachnlp/data/embeddings/sentence_embeddings_bertje_nl.npy')

In [6]:
embeddings=loaded_embeddings_nl

## 3. Preparation for Fine-Tuning Hyperparameters

### 3.1. Generate a list of hyperparameter combinations

In [7]:
import itertools

# Generating a list of hyperparameter combinations
# UMAP hyperparameters
n_neighbors_values = [5, 10, 15, 20, 25, 30] # we are aiming for strict clusters so the n_neighbors range is relatively small
n_components_values = [2, 3, 4, 5]

# HDBSCAN hyperparameters
min_cluster_size_values = [10, 15, 20, 30]
min_samples_values = [5, 10, 15, 20, 30]

combinations = list(itertools.product(n_neighbors_values, n_components_values, min_cluster_size_values, min_samples_values))

# set min_samples < min_cluster_size
filtered_combinations = [
    combination for combination in combinations if combination[3] <= combination[2] 
]

# print the total number of combinations
len(filtered_combinations)

336

### 3.2 Function to calculate intra-topic similarity

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# function to calculate intra-topic cosine similarity
def calculate_intra_topic_cosine_similarity(embeddings, doc_topics):
    topic_similarities = []
    
    for topic in set(doc_topics):
        if topic == -1:  # Skip the "outlier" topic
            continue
        
        # Get embeddings of documents in the current topic
        topic_embeddings = embeddings[np.array(doc_topics) == topic]
        
        if len(topic_embeddings) < 2:
            continue  # Skip topics with a single document
        
        # Calculate cosine similarities and average them
        cosine_sim = cosine_similarity(topic_embeddings)
        avg_cosine_sim = cosine_sim[np.triu_indices_from(cosine_sim, k=1)].mean()
        
        topic_similarities.append(avg_cosine_sim)

    return topic_similarities

### 3.3. Function to calculate silhouette scores

In [9]:
# function to calculate silhouette scores if needed (we don't use the silhouette scores for now because the clusters are probably overlapping a lot so inter-topic distance is not so informative)
from sklearn.metrics import silhouette_samples
import numpy as np

def calculate_intra_topic_silhouette_score(embeddings, topics):

    # Filter out documents assigned to the outlier topic (-1)
    mask = topics != -1
    filtered_embeddings = embeddings[mask]
    filtered_topics = topics[mask]

    # Compute silhouette scores
    silhouette_scores = silhouette_samples(filtered_embeddings, filtered_topics, metric="cosine")
    
    # Calculate average silhouette score for each topic
    unique_topics = np.unique(filtered_topics)
    avg_silhouette_per_topic = {
        topic: np.mean(silhouette_scores[filtered_topics == topic]) for topic in unique_topics
    }

    # Compute the overall average silhouette score across all topics
    avg_silhouette_score = np.mean(list(avg_silhouette_per_topic.values()))

    return avg_silhouette_score


### 3.4 Function to calculate coherence scores

In [10]:
# create the dictionary of documents
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

def generate_dictionary(documents):

    processed_docs = [doc.split() for doc in documents]
    
    # Create a Gensim dictionary
    dictionary = Dictionary(processed_docs)
    
    return dictionary, processed_docs

dictionary, processed_docs = generate_dictionary(sentences)

In [11]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary

def generate_topic_words(topics, top_n_words):
    
    topic_words = []
    
    for topic_num, words in topics.items():
        if topic_num == -1:  # Skip the outlier topic
            continue
        # Collect only the top N words for each topic
        top_words = [word for word, _ in words[:top_n_words]]
        topic_words.append(top_words)
    
    return topic_words

def calculate_coherence_score(topics, dictionary, processed_docs, coherence='c_v', top_n_words=10):

    topic_words = generate_topic_words(topics, top_n_words)

    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=processed_docs,
        dictionary=dictionary,
        coherence=coherence
    )
    
    # Step 4: Get the coherence score
    coherence_score = coherence_model.get_coherence()
    
    return coherence_score


## 4. Fine-Tune BERTje-HDBSCAN model

### 4.1 Fine-Tuning UMAP hyper parameters

In [12]:
# set the vectorizer model
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model=CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{2,}\b')


### 4.2 Function to build BERTopic model using a combination of hyperparameters

In [66]:
# build the BERTopic model
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from sklearn.cluster import AgglomerativeClustering


def build_bertopic(combination):
    n_neighbors, n_components, min_cluster_size, min_samples = combination
    cluster_model = AgglomerativeClustering(n_clusters=50)

    # Initialize BERTopic with UMAP and HDBSCAN models
    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=UMAP(n_neighbors=n_neighbors, n_components=n_components, metric='cosine', random_state=42),
        hdbscan_model=cluster_model,
        #hdbscan_model=HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, metric='euclidean', cluster_selection_method='eom', prediction_data=True),
        vectorizer_model=vectorizer_model,
        top_n_words=10,
        verbose=True
    )

    # Fit-transform to get document topics and probabilities
    doc_topics, probs = topic_model.fit_transform(sentences, embeddings)
    

    # Return only essential results
    return topic_model



### 4.3 Multiprocess for fine-tuning

In [14]:
# function to divide inputs in chunks
def chunk_list(data, chunk_size):
    for i in range(0, len(data), chunk_size):
        yield data[i:i + chunk_size]

In [16]:
# divide filtered_combinations in batches
batch_size = 20

batches = list(chunk_list(filtered_combinations, batch_size))

len(batches) # there are 17 batches

17

In [71]:
# multiprocessing to cauclate intratopic similarity
import multiprocessing
import os
num_cores = 32

batch_index=0

with multiprocessing.Pool(processes=num_cores) as pool:
    batch=batches[batch_index]
    print(f"processing batch {batch_index}")
    batch_results = pool.map(calculate_cluster_size_distribution(combination), batch)
    
    # Save each batch individually
    save_batch(batch_results, batch_index)

    # Close and delete the pool to release resources
    pool.close()
    pool.join()
    del pool  # Delete the pool variable to free memory

2024-11-07 13:43:54,882 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


processing batch 0


2024-11-07 13:44:38,894 - BERTopic - Dimensionality - Completed ✓
2024-11-07 13:44:38,900 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-11-07 13:44:48,363 - BERTopic - Cluster - Completed ✓
2024-11-07 13:44:48,381 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-11-07 13:44:49,378 - BERTopic - Representation - Completed ✓


AttributeError: 'BERTopic' object has no attribute 'get_topics_info'

#### 4.3.2. Intratopic Similarity

In [61]:
# function to save intermediate batch results
import pickle
import os

def save_batch(batch_results, batch_index):
    filename = f"/workspace/mijnidbcoachnlp/data/result_data/results_of_BERTje+HDBSCAN/batch_{batch_index}.pkl"
    with open(filename, 'wb') as file:
        pickle.dump(batch_results, file)
    print(f"Batch {batch_index} saved.")


In [22]:
# multiprocessing to cauclate intratopic similarity
import multiprocessing
import os
num_cores = 64

batch_index=0

with multiprocessing.Pool(processes=num_cores) as pool:
    batch=batches[batch_index]
    print(f"processing batch {batch_index}")
    batch_results = pool.map(calculate_cluster_size(combination), batch)
    
    # Save each batch individually
    save_batch(batch_results, batch_index)

    # Close and delete the pool to release resources
    pool.close()
    pool.join()
    del pool  # Delete the pool variable to free memory

IndexError: list index out of range

In [23]:
import pickle
import os

all_results = []
for batch_index in range(len(batches)):
    with open(f"/workspace/mijnidbcoachnlp/data/result_data/Bertje-intra-topic_similarity/batch_results_intra_topic_similarities/batch_{batch_index}.pkl", 'rb') as file:
        batch_results = pickle.load(file)
        all_results.extend(batch_results)

# Optionally separate into result_combinations and model_topic_similarities
result_combinations, model_topic_similarities = zip(*all_results)

## 5. Compare the intra-topic similarities of the 20 models with the highest intra-topic similarities (both average and spread)

In [24]:
import numpy as np
import pandas as pd

# Step 1: Calculate mean intra-topic similarity for each model
mean_similarities = [np.mean(similarities) for similarities in model_topic_similarities]

# Combine hyperparameter combinations and their corresponding mean similarities
results_df = pd.DataFrame({
    'Combination': result_combinations,
    'Mean Intra-topic Similarity': mean_similarities
})

# Sort by mean intra-topic similarity and get the top 20
top_20_results = results_df.nlargest(20, 'Mean Intra-topic Similarity')

top_20_results


,Combination,Mean Intra-topic Similarity
291,"(30, 2, 30, 15)",0.914041
265,"(25, 4, 30, 30)",0.911465
249,"(25, 3, 30, 15)",0.909481
208,"(20, 4, 30, 20)",0.909121
248,"(25, 3, 30, 10)",0.908970
330,"(30, 5, 20, 20)",0.908928
162,"(15, 5, 20, 20)",0.907228
246,"(25, 3, 20, 20)",0.907038
334,"(30, 5, 30, 20)",0.906980
307,"(30, 3, 30, 30)",0.906864


In [55]:
import multiprocessing as mp
import pandas as pd
import pickle
import numpy as np

# Function to calculate cluster size distribution for a combination
def calculate_cluster_size_distribution(combination):
    topic_model, doc_topics, probs = build_bertopic(combination)
    
    # Extract cluster sizes from topic information
    cluster_sizes = topic_model.get_topic_info()["Count"].to_list()
    
    # Calculate distribution metrics
    cluster_size_stats = {
        'Mean Cluster Size': np.mean(cluster_sizes),
        'Median Cluster Size': np.median(cluster_sizes),
        'Std Dev of Cluster Sizes': np.std(cluster_sizes),
        'Min Cluster Size': np.min(cluster_sizes),
        'Max Cluster Size': np.max(cluster_sizes),
        'Number of Clusters': len(cluster_sizes)
    }
    
    return pd.DataFrame([cluster_size_stats])

# Define function to calculate cluster size distribution and add metadata
def calculate_distribution_for_model(row):
    combination = row['Combination']
    mean_similarity = row['Mean Intra-topic Similarity']
    
    # Calculate cluster size distribution for the given combination
    cluster_size_df = calculate_cluster_size_distribution(combination)
    
    # Add metadata to the result
    cluster_size_df['Mean Intra-topic Similarity'] = mean_similarity
    cluster_size_df['Combination'] = combination
    
    return cluster_size_df

# Use multiprocessing to calculate distribution for each model in parallel
balanced_models = []
for i, row in top_20_results.iterrows():
    cluster_size_df = calculate_distribution_for_model(row)
    balanced_models.append(cluster_size_df)

# Concatenate results into a single DataFrame
balanced_models_df = pd.concat(balanced_models, ignore_index=True)

# Sort by standard deviation of cluster sizes to find the most balanced model
balanced_models_df_sorted = balanced_models_df.sort_values(by='Std Dev of Cluster Sizes')

# Save the results to a pickle file for future use
with open("/workspace/mijnidbcoachnlp/data/result_data/BERTje-cluster-distribution/balanced_models_results.pkl", "wb") as f:
    pickle.dump(balanced_models_df_sorted, f)

# Display sorted results to the user
import ace_tools as tools; tools.display_dataframe_to_user(name="Top Balanced Cluster Model Among Top 20", dataframe=balanced_models_df_sorted)

# Output the best model hyperparameters
best_model = balanced_models_df_sorted.iloc[0]
print("Best Model Hyperparameters:", best_model['Combination'])


2024-11-07 16:34:26,749 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-11-07 16:35:54,219 - BERTopic - Dimensionality - Completed ✓
2024-11-07 16:35:54,222 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-11-07 16:35:57,183 - BERTopic - Cluster - Completed ✓
2024-11-07 16:35:57,199 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-11-07 16:35:57,788 - BERTopic - Representation - Completed ✓


TypeError: cannot unpack non-iterable BERTopic object

In [70]:
topic_model = build_bertopic((30, 5, 70, 40))

2024-11-07 17:53:47,219 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2024-11-07 17:55:23,532 - BERTopic - Dimensionality - Completed ✓
2024-11-07 17:55:23,535 - BERTopic - Cluster - Start clustering the reduced embeddings
2024-11-07 17:58:05,112 - BERTopic - Cluster - Completed ✓
2024-11-07 17:58:05,125 - BERTopic - Representation - Extracting topics from clusters using representation models.
2024-11-07 17:58:05,720 - BERTopic - Representation - Completed ✓


In [68]:
cluster_sizes = topic_model.get_topic_info()["Count"].to_list()

# Calculate distribution metrics
cluster_size_stats = {
    'Mean Cluster Size': np.mean(cluster_sizes),
    'Median Cluster Size': np.median(cluster_sizes),
    'Std Dev of Cluster Sizes': np.std(cluster_sizes),
    'Min Cluster Size': np.min(cluster_sizes),
    'Max Cluster Size': np.max(cluster_sizes),
    'Number of Clusters': len(cluster_sizes)
}

cluster_size_df = pd.DataFrame([cluster_size_stats])

cluster_size_df["Combination"] = "(30, 2, 30, 15)"

cluster_size_df

,Mean Cluster Size,Median Cluster Size,Std Dev of Cluster Sizes,Min Cluster Size,Max Cluster Size,Number of Clusters,Combination
0,850.74,804.5,637.586255,26,2523,50,"(30, 2, 30, 15)"


In [71]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,0,2523,0_last_buikpijn_pijn_buik,"[last, buikpijn, pijn, buik, gevoel, darmen, d...","[Ja ik had inderdaad wat meer last., Ik heb er..."
1,1,2060,1_medicatie_weet_gaan_vraag,"[medicatie, weet, gaan, vraag, crohn, goed, ga...",[Ik weet helemaal niks van de crohn en alle vr...
2,2,1961,2_weet_goed_hoop_vind,"[weet, goed, hoop, vind, denk, gaat, gaan, ech...","[Ik weet even niet wat ik nu moet doen., Ik we..."
3,3,1844,3_prikken_vraag_verstandig_laten,"[prikken, vraag, verstandig, laten, gebruiken,...",[Ik neem aan dat ik dan nog steeds bloed moet ...
4,4,1749,4_afgesproken_afspraak_week_infuus,"[afgesproken, afspraak, week, infuus, volgende...",[Wil jij het zoals afgesproken daar naar toe s...
5,5,1723,5_klachten_voel_beter_gaat,"[klachten, voel, beter, gaat, goed, echt, merk...",[Met mijn klachten gaat het gelukkig iets bete...
6,6,1685,6_toilet_dag_wc_keer,"[toilet, dag, wc, keer, nacht, pijn, last, dag...",[Er zit nu geen bloed en slijm meer bij de ont...
7,7,1539,7_mogelijk_afspraak_telefonisch_bellen,"[mogelijk, afspraak, telefonisch, bellen, krij...",[Is het mogelijk dat deze afspraak telefonisch...
8,8,1490,8_graag_weten_contact_hoor,"[graag, weten, contact, hoor, wacht, opnemen, ...","[Dit zou ik graag weten., Graag wil ik weten o..."
9,9,1457,9_app_gebeld_bericht_telefoon,"[app, gebeld, bericht, telefoon, sorry, contac...",[ze zou mij daar donderdag voor terug bellen m...
